In [1]:
import os
import sys
import json
from pathlib import Path

os.chdir("..")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

sys.path.append("src")

import torch
import numpy as np
import pandas as pd
from torch_geometric.data import Data

In [2]:
# Input files
train_features_path = Path("data/features/df_train_features.parquet")
test_features_path = Path("data/features/df_test_features.parquet")
feature_names_path = Path("data/features/feature_names.json")
legacy_graph_path = Path("data/processed/graph_data_legacy_7f.pt")

# Output files
output_graph_path = Path("data/processed/graph_data_enriched_60f.pt")
output_feature_names_path = Path("data/processed/graph_feature_names_enriched.json")
output_summary_path = Path("reports/tables/graph_enriched_summary.csv")

# Make sure output folders exist
output_graph_path.parent.mkdir(parents=True, exist_ok=True)
output_feature_names_path.parent.mkdir(parents=True, exist_ok=True)
output_summary_path.parent.mkdir(parents=True, exist_ok=True)

In [3]:
train_df = pd.read_parquet(train_features_path)
test_df = pd.read_parquet(test_features_path)

with open(feature_names_path, "r", encoding="utf-8") as f:
    feature_names = json.load(f)

legacy_graph = torch.load(legacy_graph_path, weights_only=False)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Number of engineered features from JSON:", len(feature_names))
print("Legacy graph x shape:", tuple(legacy_graph.x.shape))
print("Legacy graph edge_index shape:", tuple(legacy_graph.edge_index.shape))
print("Legacy graph pos shape:", tuple(legacy_graph.pos.shape))

Train shape: (199167, 66)
Test shape: (60115, 66)
Number of engineered features from JSON: 60
Legacy graph x shape: (300000, 7)
Legacy graph edge_index shape: (2, 991684)
Legacy graph pos shape: (300000, 2)


In [4]:
train_df = train_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
test_df["split"] = "test"

required_columns = ["row", "col", "Burn_Prob"]
missing_train = [c for c in required_columns if c not in train_df.columns]
missing_test = [c for c in required_columns if c not in test_df.columns]

assert not missing_train, f"Missing required columns in train df: {missing_train}"
assert not missing_test, f"Missing required columns in test df: {missing_test}"

missing_features_train = [c for c in feature_names if c not in train_df.columns]
missing_features_test = [c for c in feature_names if c not in test_df.columns]

assert not missing_features_train, f"Missing engineered feature columns in train df: {missing_features_train}"
assert not missing_features_test, f"Missing engineered feature columns in test df: {missing_features_test}"

print("All required columns found.")
print("All engineered feature columns found in both train and test.")

All required columns found.
All engineered feature columns found in both train and test.


In [10]:
full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

dup_count = full_df.duplicated(subset=["row", "col"]).sum()
print("Duplicate (row, col) pairs before deduplication:", dup_count)

if dup_count > 0:
    full_df = full_df.drop_duplicates(subset=["row", "col"], keep="first").reset_index(drop=True)

print("Combined engineered dataframe shape:", full_df.shape)
print("Expected available nodes (train + test):", len(train_df) + len(test_df))

Duplicate (row, col) pairs before deduplication: 0
Combined engineered dataframe shape: (259282, 67)
Expected available nodes (train + test): 259282


In [11]:
# Build lookup from coordinate -> feature-table row index
coord_to_feature_idx = {
    (int(r), int(c)): i
    for i, (r, c) in enumerate(zip(full_df["row"].values, full_df["col"].values))
}

# Legacy graph coordinates
legacy_pos_np = legacy_graph.pos.cpu().numpy()
legacy_coords = [(int(r), int(c)) for r, c in legacy_pos_np]

# Keep only legacy nodes that exist in enriched features
kept_legacy_indices = []
feature_row_indices = []

for legacy_idx, coord in enumerate(legacy_coords):
    if coord in coord_to_feature_idx:
        kept_legacy_indices.append(legacy_idx)
        feature_row_indices.append(coord_to_feature_idx[coord])

print("Legacy graph nodes:", len(legacy_coords))
print("Available engineered nodes matched to legacy graph:", len(kept_legacy_indices))
print("Dropped legacy nodes without engineered features:", len(legacy_coords) - len(kept_legacy_indices))

Legacy graph nodes: 300000
Available engineered nodes matched to legacy graph: 259282
Dropped legacy nodes without engineered features: 40718


In [12]:
aligned_df = full_df.iloc[feature_row_indices].reset_index(drop=True)

print("Aligned dataframe shape:", aligned_df.shape)

assert len(aligned_df) == len(kept_legacy_indices), "Aligned dataframe and kept node list must match."
assert aligned_df[["row", "col"]].isna().sum().sum() == 0, "Missing row/col found after alignment."

Aligned dataframe shape: (259282, 67)


In [13]:
old_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(kept_legacy_indices)}

edge_index_np = legacy_graph.edge_index.cpu().numpy()
src_old = edge_index_np[0]
dst_old = edge_index_np[1]

edge_mask = np.isin(src_old, kept_legacy_indices) & np.isin(dst_old, kept_legacy_indices)

filtered_src_old = src_old[edge_mask]
filtered_dst_old = dst_old[edge_mask]

filtered_src_new = np.array([old_to_new[idx] for idx in filtered_src_old], dtype=np.int64)
filtered_dst_new = np.array([old_to_new[idx] for idx in filtered_dst_old], dtype=np.int64)

filtered_edge_index = np.vstack([filtered_src_new, filtered_dst_new])

print("Original number of edges:", edge_index_np.shape[1])
print("Filtered number of edges:", filtered_edge_index.shape[1])
print("Number of kept nodes:", len(kept_legacy_indices))
print("Average directed edges per kept node:", filtered_edge_index.shape[1] / len(kept_legacy_indices))

Original number of edges: 991684
Filtered number of edges: 857640
Number of kept nodes: 259282
Average directed edges per kept node: 3.3077498630834383


In [14]:
X = aligned_df[feature_names].to_numpy(dtype=np.float32)
y = aligned_df["Burn_Prob"].to_numpy(dtype=np.float32).reshape(-1, 1)
pos = aligned_df[["row", "col"]].to_numpy(dtype=np.float32)

train_mask = (aligned_df["split"].values == "train")
test_mask = (aligned_df["split"].values == "test")
val_mask = np.zeros(len(aligned_df), dtype=bool)  # placeholder for now

print("X shape:", X.shape)
print("y shape:", y.shape)
print("pos shape:", pos.shape)
print("Train nodes:", int(train_mask.sum()))
print("Test nodes:", int(test_mask.sum()))
print("Val nodes:", int(val_mask.sum()))

assert X.shape[0] == y.shape[0] == pos.shape[0] == len(kept_legacy_indices), "Node count mismatch."
assert X.shape[1] == len(feature_names), "Feature dimension mismatch."
assert np.isfinite(X).all(), "Non-finite values found in X."
assert np.isfinite(y).all(), "Non-finite values found in y."
assert np.isfinite(pos).all(), "Non-finite values found in pos."

X shape: (259282, 60)
y shape: (259282, 1)
pos shape: (259282, 2)
Train nodes: 199167
Test nodes: 60115
Val nodes: 0


In [15]:
graph_data_enriched = Data(
    x=torch.tensor(X, dtype=torch.float32),
    edge_index=torch.tensor(filtered_edge_index, dtype=torch.long),
    y=torch.tensor(y, dtype=torch.float32),
    pos=torch.tensor(pos, dtype=torch.float32),
)

graph_data_enriched.train_mask = torch.tensor(train_mask, dtype=torch.bool)
graph_data_enriched.val_mask = torch.tensor(val_mask, dtype=torch.bool)
graph_data_enriched.test_mask = torch.tensor(test_mask, dtype=torch.bool)

graph_data_enriched.feature_names = feature_names
graph_data_enriched.target_name = "Burn_Prob"
graph_data_enriched.num_valid_nodes = int(X.shape[0])
graph_data_enriched.num_edges = int(filtered_edge_index.shape[1])
graph_data_enriched.graph_version = "enriched_60f_train_test_only_phase_5_5"
graph_data_enriched.source_feature_files = [
    str(train_features_path),
    str(test_features_path),
    str(feature_names_path),
]
graph_data_enriched.legacy_graph_source = str(legacy_graph_path)

print(graph_data_enriched)

Data(x=[259282, 60], edge_index=[2, 857640], y=[259282, 1], pos=[259282, 2], train_mask=[259282], val_mask=[259282], test_mask=[259282], feature_names=[60], target_name='Burn_Prob', num_valid_nodes=259282, num_edges=857640, graph_version='enriched_60f_train_test_only_phase_5_5', source_feature_files=[3], legacy_graph_source='data\processed\graph_data_legacy_7f.pt')


In [1]:
import os
print(os.getcwd())

d:\wildfire-uncertainty-gnn\notebooks


In [3]:
from pathlib import Path
import torch

graph_path = Path("../data/processed/graph_data_enriched.pt")

g = torch.load(graph_path, weights_only=False)

print("Nodes:    ", g.num_nodes)
print("Features: ", g.num_node_features)
print("Edges:    ", g.num_edges)
print("Train:    ", g.train_mask.sum().item())
print("Val:      ", g.val_mask.sum().item())
print("Test:     ", g.test_mask.sum().item())
print(g)

Nodes:     300000
Features:  58
Edges:     991684
Train:     264456
Val:       11357
Test:      24187
Data(x=[300000, 58], edge_index=[2, 991684], y=[300000, 1], pos=[300000, 2], train_mask=[300000], val_mask=[300000], test_mask=[300000], num_nodes=300000)


# Phase 5.4 — Enriched Graph Reconstruction (Completed)

## Problem diagnosed

The legacy graph (`graph_data_legacy_7f.pt`) had three structural problems:

| Problem | Legacy graph | Phase 5.4 fix |
|---|---|---|
| Missing nodes | 259,282 of 300,000 (40,718 dropped) | All 300,000 nodes included |
| Validation mask | `val_mask = 0` (placeholder) | 11,357 val nodes (spatially disjoint) |
| Feature dimension | 7 raw raster features | 58 engineered features |

The 40,718 missing nodes occurred because `baseline_splits_spatial.npz` only indexed
259,282 nodes. The graph construction did a coordinate join against this split,
silently dropping every node not present in it.

## What Phase 5.4 did

**Script**: `phase5_4_rebuild.py --config feature_config.yaml`

### Step 1 — Full node load
Loaded all 300,000 rows from `data/processed/baseline_dataset.csv`.
Standardised column names (removed `.img` suffix, renamed `target` → `Burn_Prob`).

### Step 2 — Coordinate derivation
Derived projected (x, y) coordinates in EPSG:2100 for every node
using `data/interim/aligned/Burn_Prob.img` as the reference raster transform.

### Step 3 — DEM terrain features
Extracted 5 terrain features from `data/interim/aligned/dem_greece.tif`:
- `dem_elevation_m`, `dem_slope_deg`, `dem_aspect_sin`, `dem_aspect_cos`, `dem_twi`

Critical fix applied: DEM is EPSG:4326 (degrees). Resolution was converted
from `0.001°` to `111 m` before computing `np.gradient`, fixing the slope=90° bug.
Result: slope mean = 11.15° (physically realistic for Greece).

### Step 4 — Feature transform pipeline (all 300k nodes)
Applied the full pipeline to all 300,000 nodes simultaneously using the
complete 1898×1886 raster grid. Previous runs only saw 259,282 nodes,
causing zero-padded multi-scale stats for missing nodes.

Feature groups produced:

| Group | Count |
|---|---|
| Base raster (CFL, FSP, Ignition, Struct) | 4 |
| DEM terrain | 5 |
| NDVI (proxy from slope) | 1 |
| Fuel model one-hot | 21 |
| Interaction terms | 3 |
| Multi-scale stats (3×3, 7×7, 15×15) | 18 |
| Spatial gradients | 6 |
| **Total** | **58** |

### Step 5 — Target transformation
Applied `QuantileTransformer(output_distribution='normal')` to `Burn_Prob`.
Maps heavily right-skewed burn probability to near-Gaussian (Q-Q r²=0.998).
Transformer saved to `data/features/target_transformer.pkl`.

### Step 6 — Spatially disjoint 3-way split
Created a geographic block split by raster row (north→south):

```
Row 0      ┌──────────────────────┐  north
           │   TRAIN  (rows 0–1327)│  264,456 nodes  88.2%
Row 1328   ├──────────────────────┤
           │   VAL  (rows 1328–1517)│  11,357 nodes   3.8%
Row 1518   ├──────────────────────┤
           │   TEST  (rows 1518–1897)│  24,187 nodes   8.1%
Row 1897   └──────────────────────┘  south
```

No overlap between any split. All 300,000 nodes covered.
Split indices saved to `data/features/splits_phase54.npz`.

### Step 7 — 8-connected pixel grid edges
Built edges by matching each node's (row, col) to its 8 spatial neighbours.
All 300,000 nodes are in the lookup — no dropped nodes.

Result: 991,684 edges (avg 3.3 per node, boundary nodes have fewer).

### Step 8 — Graph assembly and save

```python
graph = Data(
    x          = torch.float32  # shape (300000, 58)
    y          = torch.float32  # shape (300000, 1)  — quantile-transformed
    pos        = torch.float32  # shape (300000, 2)  — (row, col)
    edge_index = torch.long     # shape (2, 991684)
    train_mask = torch.bool     # 264,456 True
    val_mask   = torch.bool     #  11,357 True  ← was 0 in legacy graph
    test_mask  = torch.bool     #  24,187 True
)
```

Saved to: `data/processed/graph_data_enriched.pt` (85.8 MB)

## Output files

| File | Description |
|---|---|
| `data/processed/graph_data_enriched.pt` | Final PyG graph — use this for all GNN training |
| `data/features/df_all_features.parquet` | All 300k nodes with 58 features |
| `data/features/df_train_features.parquet` | Train split (264,456 rows) |
| `data/features/df_val_features.parquet` | Val split (11,357 rows) |
| `data/features/df_test_features.parquet` | Test split (24,187 rows) |
| `data/features/feature_names.json` | Ordered list of 58 feature names |
| `data/features/splits_phase54.npz` | train_idx, val_idx, test_idx arrays |
| `data/features/target_transformer.pkl` | Fitted QuantileTransformer for inverse transform |
| `data/features/correlation_report_phase54.csv` | Pearson r per feature vs target (train only) |

## Verification

```python
import torch
g = torch.load("data/processed/graph_data_enriched.pt")
print(g.num_nodes)          # 300000
print(g.num_node_features)  # 58
print(g.num_edges)          # 991684
print(g.train_mask.sum())   # 264456
print(g.val_mask.sum())     # 11357  ← fixed
print(g.test_mask.sum())    # 24187
```

## Next step — Phase 5.5

Load `graph_data_enriched.pt` in `04_gnn_experiments.ipynb`:

```python
graph = torch.load("data/processed/graph_data_enriched.pt")
input_dim = graph.num_node_features  # 58, not 7
```

Target: GraphSAGE with 58 features should achieve test R² > 0
under the hard geographic split. Ridge baseline: −0.80 (linear).
GNN with message passing expected: R² = 0.35–0.50.